# Experiment: Phase 1 Full Run

이 노트북은 문서 참조 구조 비교 실험의 Phase 1 풀런을 쉽게 실행하기 위한 런처입니다.

실행 흐름은 `Preflight -> CLI/Model/Reasoning Smoke Test -> DryRun -> Full Run -> Aggregate -> Result Review` 입니다. 실제 풀런 전에 Codex CLI, 선택 모델, reasoning 수준이 동작하는지 최소 호출로 확인합니다.


## 00 Configuration

기본값은 `codex` + `gpt-5.4-mini` + `reasoning=medium`입니다. 필요하면 이 셀의 값만 바꾼 뒤 위에서부터 다시 실행하세요.

`REASONING_EFFORT = "default"`로 두면 agent 기본값을 쓰고 reasoning CLI 옵션을 전달하지 않습니다.


In [ ]:
from __future__ import annotations

import json
import os
import platform
import shutil
import subprocess
import tempfile
import textwrap
from datetime import datetime
from pathlib import Path

AGENT = "codex"
MODEL = "gpt-5.4-mini"
REASONING_EFFORT = "low"  # default|minimal|low|medium|high|xhigh
CODEX_REASONING_FLAG = "--reasoning-effort"  # runner 내부에서 codex 최신 CLI config 방식으로 자동 매핑
CONDITIONS = "C0,C1,C2,C3,C4"
REPEATS = 3
TASKS = "task-1-todo-crud,task-2-jwt,task-3-pagination"
TIME_BUDGET_MINUTES = 360
FAILURE_THRESHOLD = 6
CODEX_SMOKE_TIMEOUT_SEC = 60
RUN_LIVE_CODEX_SMOKE = True  # 실제 codex exec 최소 호출로 계정/모델 접근까지 검증
VALID_REASONING_EFFORTS = {"default", "minimal", "low", "medium", "high", "xhigh"}

if REASONING_EFFORT not in VALID_REASONING_EFFORTS:
    raise ValueError(f"REASONING_EFFORT must be one of {sorted(VALID_REASONING_EFFORTS)}")
if REASONING_EFFORT != "default" and AGENT != "codex":
    raise ValueError("REASONING_EFFORT is currently wired only for AGENT='codex'. Use 'default' for other agents.")

def find_repo_root() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents, Path(r"C:\git\docs_experiment")]
    for candidate in candidates:
        runner = candidate / "experiments" / "runner" / "run_experiment.ps1"
        if runner.exists():
            return candidate.resolve()
    raise RuntimeError("repo root를 찾지 못했습니다. 노트북을 docs_experiment 레포 안에서 실행하세요.")

REPO_ROOT = find_repo_root()
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_ROOT = REPO_ROOT / "experiments" / "results" / RUN_ID
RUNNER = REPO_ROOT / "experiments" / "runner" / "run_experiment.ps1"
MONITOR = REPO_ROOT / "experiments" / "runner" / "monitor_quota.ps1"
AGGREGATE = REPO_ROOT / "experiments" / "runner" / "aggregate.py"
VENV_PY = REPO_ROOT / ".venv" / "Scripts" / "python.exe"
SAFE_MODEL = MODEL.translate(str.maketrans({c: "_" for c in '\\/:*?"<>|'}))
SAFE_REASONING = REASONING_EFFORT.translate(str.maketrans({c: "_" for c in '\\/:*?"<>|'}))
AGENT_ROOT = RUN_ROOT / AGENT / SAFE_MODEL
if REASONING_EFFORT != "default":
    AGENT_ROOT = AGENT_ROOT / f"reasoning-{SAFE_REASONING}"

CONDITION_LIST = [x.strip() for x in CONDITIONS.split(",") if x.strip()]
TASK_LIST = [x.strip() for x in TASKS.split(",") if x.strip()]
EXPECTED_RUNS = len(CONDITION_LIST) * REPEATS * len(TASK_LIST)

POWERSHELL = shutil.which("pwsh") or shutil.which("powershell") or "powershell"

print(f"repo root       : {REPO_ROOT}")
print(f"run id          : {RUN_ID}")
print(f"agent/model     : {AGENT} / {MODEL}")
print(f"reasoning       : {REASONING_EFFORT}")
print(f"conditions      : {CONDITIONS}")
print(f"repeats         : {REPEATS}")
print(f"tasks           : {TASKS}")
print(f"expected runs   : {EXPECTED_RUNS} ({len(CONDITION_LIST)} conditions x {REPEATS} repeats x {len(TASK_LIST)} tasks)")
print(f"run root        : {RUN_ROOT}")
print(f"agent root      : {AGENT_ROOT}")
print(f"powershell      : {POWERSHELL}")


## Shared Helpers

아래 헬퍼는 명령 실행, 에러 분류, reasoning guard, full-run 보호 플래그를 제공합니다.


In [ ]:
CLI_PREFLIGHT_OK = False
MODEL_SMOKE_OK = False
REASONING_SMOKE_OK = False

class PreflightError(RuntimeError):
    pass

def make_run_env():
    env = os.environ.copy()
    scripts = str(VENV_PY.parent)
    env["PATH"] = scripts + os.pathsep + env.get("PATH", "")
    return env

def run_cmd(args, *, cwd=REPO_ROOT, timeout=None, check=False, env=None):
    args = [str(a) for a in args]
    print("$ " + " ".join(args))
    completed = subprocess.run(
        args,
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        timeout=timeout,
        env=env,
    )
    if completed.stdout.strip():
        print(completed.stdout)
    if completed.stderr.strip():
        print(completed.stderr)
    if check and completed.returncode != 0:
        raise RuntimeError(f"command failed with exit code {completed.returncode}")
    return completed

def stream_cmd(args, *, cwd=REPO_ROOT, env=None):
    args = [str(a) for a in args]
    print("$ " + " ".join(args))
    proc = subprocess.Popen(
        args,
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=env,
        bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
    return proc.wait()

def classify_codex_failure(text: str) -> str:
    lower = text.lower()
    if "optional dependency" in lower or "codex-win32-x64" in lower:
        return "Codex CLI 설치/optional dependency 문제"
    if "not recognized" in lower or "command not found" in lower or ("not found" in lower and "model" not in lower):
        return "Codex CLI 설치/PATH 문제"
    if "api key" in lower or "unauthorized" in lower or "authentication" in lower or "login" in lower or "not logged in" in lower:
        return "인증 문제"
    if "trusted directory" in lower or "skip-git-repo-check" in lower or "not inside a trusted" in lower:
        return "Codex CLI trust/git repo check 문제"
    if "reasoning" in lower or "effort" in lower or CODEX_REASONING_FLAG.lower() in lower:
        return "reasoning 수준 또는 CLI 옵션 지원 문제"
    if "model" in lower or "unsupported" in lower or "does not exist" in lower or "access" in lower:
        return "모델명 또는 모델 접근 권한 문제"
    return "기타 실행 문제"

def fail_with_codex_help(result, context: str):
    combined = "\n".join([result.stdout or "", result.stderr or ""]).strip()
    kind = classify_codex_failure(combined)
    print(f"[{context}] 실패 분류: {kind}")
    if "optional dependency" in kind:
        print("복구 명령: npm install -g @openai/codex@latest")
    if "reasoning" in kind:
        print(f"확인 필요: Codex CLI가 `{CODEX_REASONING_FLAG} {REASONING_EFFORT}`를 지원하는지 확인하세요.")
    raise PreflightError(f"{context} failed: {kind}")

def codex_exec_command(prompt: str, *, include_reasoning: bool):
    # codex exec 는 stdin EOF를 받아야 대기 상태로 멈추지 않으므로 빈 라인을 파이프한다.
    parts = ['"" | codex exec --skip-git-repo-check', f'--model "{MODEL}"']
    if include_reasoning and REASONING_EFFORT != "default":
        # 최신 codex-cli는 --reasoning-effort 대신 config override를 사용
        parts.append(f'-c \'model_reasoning_effort="{REASONING_EFFORT}"\'')
    parts.append('--json')
    parts.append(f'"{prompt}"')
    return " ".join(parts)

def require_smoke_ok():
    if not CLI_PREFLIGHT_OK:
        raise PreflightError("01 Environment Preflight 셀을 먼저 성공시켜야 합니다.")
    if not MODEL_SMOKE_OK:
        raise PreflightError("02 Model/Reasoning Smoke Test 셀의 모델 검증을 먼저 성공시켜야 합니다.")
    if not REASONING_SMOKE_OK:
        raise PreflightError("02 Model/Reasoning Smoke Test 셀의 reasoning 검증을 먼저 성공시켜야 합니다.")


## 01 Environment Preflight

Python 3.11+ 가상환경, requirements 설치, Codex CLI 실행 가능 여부를 확인합니다. 현재 로컬에서 관찰된 `@openai/codex-win32-x64` optional dependency 누락도 여기서 풀런 전에 잡습니다.


In [ ]:
# cell 6 : Environment Preflight
required_paths = [RUNNER, MONITOR, AGGREGATE, REPO_ROOT / "requirements.txt"]
for path in required_paths:
    if not path.exists():
        raise PreflightError(f"필수 파일이 없습니다: {path}")

if not VENV_PY.exists():
    py_launcher = shutil.which("py")
    created = None

    # 1) 현재 노트북 커널 Python으로 먼저 시도 (가장 일관됨)
    current_python = __import__("sys").executable
    current_ver = tuple(int(x) for x in platform.python_version().split(".")[:2])
    print(f"현재 커널 Python : {current_python} ({platform.python_version()})")
    if current_ver >= (3, 11):
        print("현재 커널 Python으로 .venv 생성 시도합니다.")
        created = run_cmd([current_python, "-m", "venv", str(REPO_ROOT / ".venv")], timeout=300)

    # 2) 실패 시 py launcher로 3.13/3.12/3.11 순차 시도
    if (created is None or created.returncode != 0) and py_launcher:
        for py_flag in ["-3.13", "-3.12", "-3.11"]:
            print(f".venv가 없어 {py_flag}로 생성 시도합니다.")
            created = run_cmd([py_launcher, py_flag, "-m", "venv", str(REPO_ROOT / ".venv")], timeout=300)
            if created.returncode == 0:
                break

    # 3) 마지막 fallback: python.exe PATH
    if created is None or created.returncode != 0:
        python_exe = shutil.which("python")
        if python_exe:
            print("fallback: `python -m venv`로 생성 시도합니다.")
            created = run_cmd([python_exe, "-m", "venv", str(REPO_ROOT / ".venv")], timeout=300)

    if created is None or created.returncode != 0:
        detected = ""
        if py_launcher:
            versions = run_cmd([py_launcher, "-0p"], timeout=30)
            detected = (versions.stdout or versions.stderr or "").strip()
        msg = (
            "Python 3.11+ 기반 .venv 생성에 실패했습니다.\n"
            "현재 py -0p에서 3.11+가 없거나 선택되지 않았습니다.\n"
            "권장: Python 3.11 이상 설치 후 재시도\n"
            "확인: py -0p\n"
            f"탐지 결과:\n{detected}"
        )
        raise PreflightError(msg)

version_result = run_cmd([VENV_PY, "-c", "import sys; print(f'{sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}')"], check=True)
version = tuple(int(x) for x in version_result.stdout.strip().split(".")[:2])
if version < (3, 11):
    raise PreflightError(f".venv Python 버전이 낮습니다: {version_result.stdout.strip()} (필요: 3.11+)")

print("requirements.txt를 설치/갱신합니다.")
run_cmd([VENV_PY, "-m", "pip", "install", "-r", REPO_ROOT / "requirements.txt"], timeout=900, check=True)

codex_path = shutil.which("codex")
if not codex_path:
    raise PreflightError("codex CLI를 찾지 못했습니다. 설치: npm install -g @openai/codex@latest")
print(f"codex path      : {codex_path}")

codex_version = run_cmd(
    [POWERSHELL, "-NoProfile", "-ExecutionPolicy", "Bypass", "-Command", "codex --version"],
    timeout=30,
)
if codex_version.returncode != 0:
    fail_with_codex_help(codex_version, "codex --version")

if os.environ.get("OPENAI_API_KEY"):
    print("OPENAI_API_KEY  : set")
else:
    print("OPENAI_API_KEY  : not set; Codex CLI 로그인/설정으로 인증되어 있을 수 있습니다. 다음 smoke test에서 검증합니다.")

CLI_PREFLIGHT_OK = True
print("Environment preflight OK")


## 02 Native Model/Reasoning Guard

`codex debug models`로 Codex CLI가 제공하는 native 모델 카탈로그를 읽어 `MODEL`과 `REASONING_EFFORT` 지원 여부를 확인합니다. 기본값은 실제 모델 호출을 하지 않으므로 토큰을 쓰지 않습니다.

`RUN_LIVE_CODEX_SMOKE = True`가 기본값이므로 `codex exec --skip-git-repo-check` 최소 호출까지 수행해 인증/런타임도 검증합니다.


In [ ]:
if not CLI_PREFLIGHT_OK:
    raise PreflightError("01 Environment Preflight 셀을 먼저 실행하세요.")

debug_cmd = [
    POWERSHELL,
    "-NoProfile",
    "-ExecutionPolicy",
    "Bypass",
    "-Command",
    "codex debug models",
]
print("$ " + " ".join(str(x) for x in debug_cmd))
models_result = subprocess.run(
    [str(x) for x in debug_cmd],
    cwd=str(REPO_ROOT),
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    timeout=60,
)
if models_result.returncode != 0:
    fail_with_codex_help(models_result, "codex debug models")

try:
    model_catalog = json.loads(models_result.stdout)
except json.JSONDecodeError as exc:
    print((models_result.stdout or "")[:2000])
    raise PreflightError(f"codex debug models JSON 파싱 실패: {exc}") from exc

models = model_catalog.get("models", [])
by_slug = {m.get("slug"): m for m in models if m.get("slug")}
if MODEL not in by_slug:
    visible = sorted(k for k in by_slug if k)
    nearby = [m for m in visible if MODEL.split("-")[0] in m][:20]
    raise PreflightError(
        f"Codex native catalog에 MODEL={MODEL!r}가 없습니다. "
        f"유사 후보: {nearby or visible[:20]}"
    )

model_info = by_slug[MODEL]
supported_reasoning = [
    item.get("effort")
    for item in (model_info.get("supported_reasoning_levels") or [])
    if item.get("effort")
]
default_reasoning = model_info.get("default_reasoning_level") or "unknown"

if REASONING_EFFORT != "default" and REASONING_EFFORT not in supported_reasoning:
    raise PreflightError(
        f"MODEL={MODEL!r}는 REASONING_EFFORT={REASONING_EFFORT!r}를 지원하지 않습니다. "
        f"지원값: {supported_reasoning or ['default only']}, 기본값: {default_reasoning}"
    )

MODEL_SMOKE_OK = True
REASONING_SMOKE_OK = True
print(f"Native model catalog OK: {MODEL}")
print(f"Supported reasoning    : {supported_reasoning or ['default only']}")
print(f"Selected reasoning     : {REASONING_EFFORT} (model default: {default_reasoning})")

if RUN_LIVE_CODEX_SMOKE:
    print("RUN_LIVE_CODEX_SMOKE=True: 실제 codex exec 최소 호출을 수행합니다.")
    with tempfile.TemporaryDirectory(prefix="codex-model-smoke-") as tmp:
        model_smoke = run_cmd(
            [
                POWERSHELL,
                "-NoProfile",
                "-ExecutionPolicy",
                "Bypass",
                "-Command",
                codex_exec_command("Reply with MODEL_OK only.", include_reasoning=False),
            ],
            cwd=Path(tmp),
            timeout=CODEX_SMOKE_TIMEOUT_SEC,
        )

    model_output = "\n".join([model_smoke.stdout or "", model_smoke.stderr or ""]).strip()
    if model_smoke.returncode != 0:
        fail_with_codex_help(model_smoke, f"live model smoke test ({MODEL})")
    if not model_output:
        raise PreflightError("live model smoke test가 성공 exit code를 반환했지만 출력이 비어 있습니다.")

    if REASONING_EFFORT != "default":
        with tempfile.TemporaryDirectory(prefix="codex-reasoning-smoke-") as tmp:
            reasoning_smoke = run_cmd(
                [
                    POWERSHELL,
                    "-NoProfile",
                    "-ExecutionPolicy",
                    "Bypass",
                    "-Command",
                    codex_exec_command("Reply with REASONING_OK only.", include_reasoning=True),
                ],
                cwd=Path(tmp),
                timeout=CODEX_SMOKE_TIMEOUT_SEC,
            )

        reasoning_output = "\n".join([reasoning_smoke.stdout or "", reasoning_smoke.stderr or ""]).strip()
        if reasoning_smoke.returncode != 0:
            fail_with_codex_help(reasoning_smoke, f"live reasoning smoke test ({MODEL}, {REASONING_EFFORT})")
        if not reasoning_output:
            raise PreflightError("live reasoning smoke test가 성공 exit code를 반환했지만 출력이 비어 있습니다.")

    print("Live Codex exec smoke OK")
else:
    print("Live Codex exec smoke skipped. Full Run은 native codex exec로 실행됩니다.")


## 03 DryRun

실제 에이전트 호출과 평가를 생략하고 45 run 매트릭스가 올바르게 구성되는지 확인합니다.


In [ ]:
require_smoke_ok()

dry_cmd = [
    POWERSHELL,
    "-NoProfile",
    "-ExecutionPolicy",
    "Bypass",
    "-File",
    RUNNER,
    "-RunId",
    RUN_ID,
    "-Agent",
    AGENT,
    "-Model",
    MODEL,
    "-ReasoningEffort",
    REASONING_EFFORT,
    "-ReasoningFlag",
    CODEX_REASONING_FLAG,
    "-Conditions",
    CONDITIONS,
    "-Repeats",
    str(REPEATS),
    "-Tasks",
    TASKS,
    "-DryRun",
    "-NoEvaluate",
]
dry = run_cmd(dry_cmd, timeout=600, env=make_run_env())
if dry.returncode != 0:
    raise RuntimeError("DryRun이 실패했습니다. 위 로그를 확인하세요.")

runs_csv = AGENT_ROOT / "runs.csv"
if not runs_csv.exists():
    raise RuntimeError(f"DryRun runs.csv가 생성되지 않았습니다: {runs_csv}")
print(f"DryRun OK: {runs_csv}")


## 04 Full Run

실제 Phase 1 풀런입니다. 예상 규모는 `5 조건 x 3 반복 x 3 task = 45 run`입니다. 장시간 실행될 수 있습니다.


In [ ]:
require_smoke_ok()

monitor_cmd = [
    str(POWERSHELL),
    "-NoProfile",
    "-ExecutionPolicy",
    "Bypass",
    "-File",
    str(MONITOR),
    "-RunRoot",
    str(RUN_ROOT),
    "-TimeBudgetMinutes",
    str(TIME_BUDGET_MINUTES),
    "-FailureThreshold",
    str(FAILURE_THRESHOLD),
]
print("Quota monitor는 백그라운드로 자동 실행하지 않습니다.")
print("필요하면 별도 PowerShell 터미널에서 아래 명령을 실행하세요:")
print(" ".join(monitor_cmd))

full_cmd = [
    POWERSHELL,
    "-NoProfile",
    "-ExecutionPolicy",
    "Bypass",
    "-File",
    RUNNER,
    "-RunId",
    RUN_ID,
    "-Agent",
    AGENT,
    "-Model",
    MODEL,
    "-ReasoningEffort",
    REASONING_EFFORT,
    "-ReasoningFlag",
    CODEX_REASONING_FLAG,
    "-Conditions",
    CONDITIONS,
    "-Repeats",
    str(REPEATS),
    "-Tasks",
    TASKS,
]
exit_code = stream_cmd(full_cmd, env=make_run_env())
if exit_code != 0:
    raise RuntimeError(f"Full run failed with exit code {exit_code}")
print("Full run OK")


## 05 Aggregate

전체 `RunId` 아래의 모든 `metrics.json`을 모아 `summary.csv`와 `report.md`를 생성합니다.


In [ ]:
if not RUN_ROOT.exists():
    raise RuntimeError(f"run root가 없습니다: {RUN_ROOT}")

agg_cmd = [VENV_PY, AGGREGATE, "--run-root", RUN_ROOT]
exit_code = stream_cmd(agg_cmd)
if exit_code != 0:
    raise RuntimeError(f"aggregate.py failed with exit code {exit_code}")

print(f"summary.csv: {RUN_ROOT / 'summary.csv'}")
print(f"report.md  : {RUN_ROOT / 'report.md'}")


## 06 Review Results

`summary.csv`를 읽어 조건별 핵심 지표 평균을 확인합니다.


In [ ]:
import sys
from IPython.display import Markdown, display

try:
    import pandas as pd
except ImportError:
    print("현재 Jupyter 커널에 pandas가 없어 현재 커널에 pandas/tabulate를 설치합니다.")
    run_cmd([sys.executable, "-m", "pip", "install", "pandas", "tabulate"], timeout=900, check=True)
    import pandas as pd

summary_path = RUN_ROOT / "summary.csv"
report_path = RUN_ROOT / "report.md"
if not summary_path.exists():
    scoped_summary = AGENT_ROOT / "summary.csv"
    if scoped_summary.exists():
        summary_path = scoped_summary
    else:
        raise RuntimeError(f"summary.csv를 찾지 못했습니다: {summary_path}")

df = pd.read_csv(summary_path)
print(f"loaded rows: {len(df)}")
display(df.head())

metrics = [
    "requirements_fulfillment",
    "test_pass_rate",
    "design_alignment",
    "static_errors_total",
    "reprompt_count",
    "wall_seconds",
    "total_tokens",
    "coverage_percent",
]
available = [m for m in metrics if m in df.columns]
group_cols = [c for c in ["agent", "model", "reasoning_effort", "condition"] if c in df.columns]
if available and group_cols:
    pivot = (
        df.groupby(group_cols, dropna=False)[available]
        .mean(numeric_only=True)
        .round(3)
        .reset_index()
    )
    display(pivot)

if report_path.exists():
    display(Markdown(f"[Open aggregate report]({report_path.as_posix()})"))
scoped_report = AGENT_ROOT / "report.md"
if scoped_report.exists():
    display(Markdown(f"[Open scoped report]({scoped_report.as_posix()})"))

print(f"run root  : {RUN_ROOT}")
print(f"agent root: {AGENT_ROOT}")
